# Z_2_1_Enriching_Dataset

## Objective:

This Jupyter Notebook aims to enrich the formatted SEO dataset by integrating essential financial, macroeconomic, and firm-level indicators. The goal is to prepare a modeling-ready dataset that incorporates:

- Daily close prices, EPS, and dividend information (from WRDS security files)

- Annual financial ratios (e.g., ROA, ROE, margins) using CUSIP and ticker-based mappings

- Macroeconomic factors including:

    - U.S. Treasury yields (1-year and 30-day)

    - Consumer Price Index (CPI)

    - Projected GDP growth rate (FRED GDPC1CTMLR)

- Time alignment logic using lagged_Offer_Date, matched to trading calendars and fiscal reporting quarters

This notebook also performs:

- Custom CUSIP-9 generation from CUSIP-8 for accurate merging

- Duplicate handling and deduplication in annual ratios

- Logical imputation from multiple identifier sources (CUSIP9 and Ticker)

### Using security daily data containing prccd to impute missing close price values in SEO ISSUES

In [34]:
# Installing pandas_market_calendars package
#!pip install pandas_market_calendars

In [35]:
import pandas as pd
import zipfile
import os
import pandas_market_calendars as mcal



In [36]:
# ===========================================
# 1. Import Libraries and Load External Price Data
# ===========================================

# Load zipped external daily security data used for imputing missing close prices
file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\security_data_using_issue_ticker.zip'

with zipfile.ZipFile(file_path) as z:
    with z.open('meabriv7ju9f5juy.csv') as f:
        ext_close_price = pd.read_csv(f)

# Inspect structure of external data
ext_close_price.info()
ext_close_price.head(10)


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\983950071.py:10: DtypeWarning: Columns (5,15) have mixed types. Specify dtype option on import or set low_memory=False.
  ext_close_price = pd.read_csv(f)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7773604 entries, 0 to 7773603
Data columns (total 17 columns):
 #   Column    Dtype  
---  ------    -----  
 0   GVKEY     int64  
 1   LPERMNO   int64  
 2   iid       int64  
 3   datadate  object 
 4   tic       object 
 5   cusip     object 
 6   cshoc     float64
 7   cshtrd    float64
 8   dvi       float64
 9   eps       float64
 10  prccd     float64
 11  trfd      float64
 12  cik       float64
 13  idbflag   object 
 14  sic       int64  
 15  spcsrc    object 
 16  state     object 
dtypes: float64(7), int64(4), object(6)
memory usage: 1008.2+ MB


,GVKEY,LPERMNO,iid,datadate,tic,cusip,cshoc,cshtrd,dvi,eps,prccd,trfd,cik,idbflag,sic,spcsrc,state
0,1045,21020,4,2015-04-02,AAL,02376R102,696650000.0,13917960.0,0.4,4.02,49.175,1.007121,6201.0,B,4512,B-,TX
1,1045,21020,4,2023-01-11,AAL,02376R102,649901000.0,28609490.0,0.0,-2.48,15.340,1.060905,6201.0,B,4512,B-,TX
2,1045,21020,4,2020-08-21,AAL,02376R102,508561000.0,35571980.0,0.0,-8.17,12.160,1.060905,6201.0,B,4512,B-,TX
3,1045,21020,4,2016-04-01,AAL,02376R102,603018000.0,11243140.0,0.4,9.38,39.520,1.016626,6201.0,B,4512,B-,TX
4,1045,21020,1,2010-08-19,AAMRQ,001765106,333050000.0,8164074.0,0.0,-3.80,6.670,2.145309,6201.0,B,4512,B-,TX
5,1045,21020,4,2018-02-26,AAL,02376R102,473139000.0,4961181.0,0.4,3.92,55.000,1.036128,6201.0,B,4512,B-,TX
6,1045,21020,4,2015-07-09,AAL,02376R102,692799000.0,8477881.0,0.4,4.70,39.670,1.009207,6201.0,B,4512,B-,TX
7,1045,21020,4,2022-09-30,AAL,02376R102,649846000.0,30046210.0,0.0,-2.95,12.040,1.060905,6201.0,B,4512,B-,TX
8,1045,21020,1,2010-07-26,AAMRQ,001765106,333050000.0,9333893.0,0.0,-3.80,7.260,2.145309,6201.0,B,4512,B-,TX
9,1045,21020,4,2020-02-06,AAL,02376R102,438058000.0,8333725.0,0.4,3.80,28.300,1.060905,6201.0,B,4512,B-,TX


In [37]:
# ===========================================
# 2. Preprocess External Data for Merge
# ===========================================

# Select only relevant columns: date, CUSIP, and close price
ext_considered = ext_close_price[['datadate', 'cusip', 'prccd']]

# Convert 'datadate' to datetime format
ext_considered['datadate'] = pd.to_datetime(ext_considered['datadate'], errors='coerce')

# Preview cleaned external dataset
ext_considered.head()


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\2285862403.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ext_considered['datadate'] = pd.to_datetime(ext_considered['datadate'], errors='coerce')


,datadate,cusip,prccd
0,2015-04-02,02376R102,49.175
1,2023-01-11,02376R102,15.340
2,2020-08-21,02376R102,12.160
3,2016-04-01,02376R102,39.520
4,2010-08-19,001765106,6.670


In [38]:
# ===========================================
# 3. Load the Formatted SEO Dataset
# ===========================================

# Define path to formatted SEO dataset
output_path = r'data\Original SEO ISSUES Dataset RAW'
parquet_filename = r'SEO_ISSUES_Formatted.parquet'
full_path = os.path.join(output_path, parquet_filename)

# Read preprocessed SEO dataset
original_seo = pd.read_parquet(full_path)

# Inspect structure and preview data
original_seo.info()
original_seo.head(5)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13801 entries, 0 to 13800
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13801 non-null  object        
 1   Price_Date_Diff               13801 non-null  int64         
 2   Issue_Date                    13801 non-null  datetime64[ns]
 3   Offer_Year                    13801 non-null  int64         
 4   New_Issues_Primary_Exchange   13724 non-null  category      
 5   Issuer_Name_Full              13801 non-null  object        
 6   Issuer_CUSIP8                 11943 non-null  object        
 7   Issuer_CUSIP9                 13801 non-null  object        
 8   New_Issues_HiTech_Group       13801 non-null  object        
 9   Issue_Type                    13801 non-null  category      
 10  Regulation_Types              12456 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

,Issuer_Ticker,Price_Date_Diff,Issue_Date,Offer_Year,New_Issues_Primary_Exchange,Issuer_Name_Full,Issuer_CUSIP8,Issuer_CUSIP9,New_Issues_HiTech_Group,Issue_Type,...,Security_Type_This_Mkt,Offer_Price_USD,Shares_Offered_All_Mkts,Shares_Offered_This_Mkt,Primary_Amt_Offered_All_Mkts,Primary_Amt_Offered_This_Mkt,Price_at_Close_Issue_Date,Price_at_Close_Offer_Date,Price_1_Day_After,Price_1_Day_Before
0,CCRE,0,2010-01-01,2010,OTC,Can-Cal Resources Ltd,13470510,134705102,Non-Hitech,Follow-On,...,Units,0.13,2966600.0,255000,0.37,0.03,NaN,NaN,NaN,NaN
1,ANX,1,2010-01-03,2010,NYSE Amex,ADVENTRX Pharmaceuticals Inc,00764X10,00764X103,Biotechnology,Convertible,...,Units,1000.00,19000.0,19000,19.00,19.00,NaN,NaN,NaN,NaN
2,TXCCQ,0,2010-01-04,2010,Nasdaq,TranSwitch Corp,89406530,894065309,Electronics;Computer Equipment,Follow-On,...,Common Stock,1.60,1950000.0,1950000,3.12,3.12,1.99,1.99,1.86,2.10
3,CVVT,0,2010-01-04,2010,Nasdaq,China Valves Technology Inc,16947620,169476207,Non-Hitech,Follow-On,...,Common Stock,9.00,2414113.0,2414113,21.73,21.73,9.40,9.40,9.45,9.24
4,TYY,0,2010-01-05,2010,New York Stock Exchange,Tortoise Energy Capital Corp,89147U10,89147U100,Non-Hitech,Follow-On,...,Common Stock,22.70,1226995.0,1226995,27.85,27.85,24.13,23.87,23.94,24.13


In [39]:
# ===========================================
# 4. Merge SEO Dataset with External Close Price Data
# ===========================================

# Merge on Issue_Date and Issuer_CUSIP9 to align with daily close prices
ext_merged = pd.merge(
    original_seo, ext_considered,
    how='left',
    left_on=['Issue_Date', 'Issuer_CUSIP9'],
    right_on=['datadate', 'cusip']
)

# Drop 'datadate' as it's now redundant
ext_merged.drop(columns=['datadate'], inplace=True)

# Inspect merged dataset
ext_merged.info()
ext_merged.head(20)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13801 entries, 0 to 13800
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13801 non-null  object        
 1   Price_Date_Diff               13801 non-null  int64         
 2   Issue_Date                    13801 non-null  datetime64[ns]
 3   Offer_Year                    13801 non-null  int64         
 4   New_Issues_Primary_Exchange   13724 non-null  category      
 5   Issuer_Name_Full              13801 non-null  object        
 6   Issuer_CUSIP8                 11943 non-null  object        
 7   Issuer_CUSIP9                 13801 non-null  object        
 8   New_Issues_HiTech_Group       13801 non-null  object        
 9   Issue_Type                    13801 non-null  category      
 10  Regulation_Types              12456 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

,Issuer_Ticker,Price_Date_Diff,Issue_Date,Offer_Year,New_Issues_Primary_Exchange,Issuer_Name_Full,Issuer_CUSIP8,Issuer_CUSIP9,New_Issues_HiTech_Group,Issue_Type,...,Shares_Offered_All_Mkts,Shares_Offered_This_Mkt,Primary_Amt_Offered_All_Mkts,Primary_Amt_Offered_This_Mkt,Price_at_Close_Issue_Date,Price_at_Close_Offer_Date,Price_1_Day_After,Price_1_Day_Before,cusip,prccd
0,CCRE,0,2010-01-01,2010,OTC,Can-Cal Resources Ltd,13470510,134705102,Non-Hitech,Follow-On,...,2966600.0,255000,0.37,0.03,NaN,NaN,NaN,NaN,NaN,NaN
1,ANX,1,2010-01-03,2010,NYSE Amex,ADVENTRX Pharmaceuticals Inc,00764X10,00764X103,Biotechnology,Convertible,...,19000.0,19000,19.00,19.00,NaN,NaN,NaN,NaN,NaN,NaN
2,TXCCQ,0,2010-01-04,2010,Nasdaq,TranSwitch Corp,89406530,894065309,Electronics;Computer Equipment,Follow-On,...,1950000.0,1950000,3.12,3.12,1.99,1.99,1.86,2.10,894065309,1.99
3,CVVT,0,2010-01-04,2010,Nasdaq,China Valves Technology Inc,16947620,169476207,Non-Hitech,Follow-On,...,2414113.0,2414113,21.73,21.73,9.40,9.40,9.45,9.24,169476207,9.40
4,TYY,0,2010-01-05,2010,New York Stock Exchange,Tortoise Energy Capital Corp,89147U10,89147U100,Non-Hitech,Follow-On,...,1226995.0,1226995,27.85,27.85,24.13,23.87,23.94,24.13,89147U100,24.13
5,ORNI,0,2010-01-05,2010,OTC,Oragenics Inc,68402310,684023104,Biotechnology,Follow-On,...,10016250.0,10016250,2.50,2.50,NaN,NaN,NaN,NaN,NaN,NaN
6,ETP,0,2010-01-06,2010,New York Stock Exchange,Energy Transfer Partners LP,29273R10,29273R109,Non-Hitech,Follow-On,...,8500000.0,8500000,380.12,380.12,NaN,45.45,45.47,45.30,NaN,NaN
7,COSI,0,2010-01-06,2010,Nasdaq,Cosi Inc,22122P10,22122P101,Non-Hitech,Follow-On,...,10000000.0,10000000,5.00,5.00,0.66,0.70,0.71,0.66,NaN,NaN
8,BZH,0,2010-01-06,2010,New York Stock Exchange,Beazer Homes USA Inc,07556Q10,07556Q105,Non-Hitech,Follow-On,...,19500000.0,19500000,89.70,89.70,4.77,5.13,5.15,4.77,NaN,NaN
9,ZGEN.2,0,2010-01-06,2010,Nasdaq,ZymoGenetics Inc,98985T10,98985T109,Biotechnology,Follow-On,...,14000000.0,14000000,84.00,84.00,6.77,6.11,6.25,6.77,98985T109,6.77


In [40]:
# ===========================================
# 5. Impute Close Prices at Issue Date
# ===========================================

# Remove columns that will be replaced or are no longer needed
ext_merged.drop(columns=['Price_1_Day_After', 'Price_1_Day_Before'], inplace=True)

# Fill missing 'Price_at_Close_Issue_Date' using external close price (prccd)
ext_merged['Price_at_Close_Issue_Date'] = ext_merged['Price_at_Close_Issue_Date'].fillna(ext_merged['prccd'])

# For rows where price date difference is 0 (same as offer date), fill missing close price using offer date price
ext_merged.loc[ext_merged['Price_Date_Diff'] == 0, 'Price_at_Close_Issue_Date'] = \
    ext_merged.loc[ext_merged['Price_Date_Diff'] == 0, 'Price_at_Close_Issue_Date'].fillna(
        ext_merged['Price_at_Close_Offer_Date']
    )

# Confirm updated structure
ext_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13801 entries, 0 to 13800
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13801 non-null  object        
 1   Price_Date_Diff               13801 non-null  int64         
 2   Issue_Date                    13801 non-null  datetime64[ns]
 3   Offer_Year                    13801 non-null  int64         
 4   New_Issues_Primary_Exchange   13724 non-null  category      
 5   Issuer_Name_Full              13801 non-null  object        
 6   Issuer_CUSIP8                 11943 non-null  object        
 7   Issuer_CUSIP9                 13801 non-null  object        
 8   New_Issues_HiTech_Group       13801 non-null  object        
 9   Issue_Type                    13801 non-null  category      
 10  Regulation_Types              12456 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

In [41]:
# ===========================================
# 6. Final Cleanup Before Further Processing
# ===========================================

# Drop columns no longer needed after imputation
ext_merged.drop(columns=['prccd', 'cusip'], inplace=True)

# Drop rows where 'Offer_Price_USD' is missing
ext_merged_new = ext_merged.dropna(subset=['Offer_Price_USD'])

# Reset index to clean up after row drops
ext_merged_new.reset_index(drop=True, inplace=True)

# Check structure after cleanup
ext_merged_new.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 24 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

In [42]:
# ===========================================
# 7. Create Offer_Date
# ===========================================

# Offer_Date = Issue_Date - Price_Date_Diff (in days)
# This reflects the date when the SEO offer was made, used for time-based calculations
ext_merged_new['Offer_Date'] = ext_merged_new['Issue_Date'] - pd.to_timedelta(
    ext_merged_new['Price_Date_Diff'], unit='D'
)

# Confirm column addition
ext_merged_new.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\3395329442.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ext_merged_new['Offer_Date'] = ext_merged_new['Issue_Date'] - pd.to_timedelta(


## Trading Day Adjustment Logic (Markdown Cell Suggestion)

The following table summarizes the rules used for aligning dates to trading days based on market context:

| **Use Case**                                                               | **Date Type** | **Adjust to**            | **Reason**                                                                                   |
|----------------------------------------------------------------------------|---------------|---------------------------|----------------------------------------------------------------------------------------------|
| Use price data just before SEO                                            | Issue Date    | Previous trading day      | Captures pre-event market signals                                                           |
| SEO announced outside market hours / on non-trading day                   | Offer Date    | Next trading day          | Market reacts when it reopens                                                               |
| Compare offer price to recent close                                       | Offer Date    | Previous trading day      | Pricing usually based on last available market close                                        |


In [43]:
# ===========================================
# 8. Compute Lagged Offer Date (1 Day Before Offer)
# ===========================================

# This is used to pull market data one day before the offer (for return calculations, etc.)
ext_merged_new['lagged_Offer_Date'] = (ext_merged_new['Offer_Date'] - pd.Timedelta(days=1)).dt.normalize()


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\2542404339.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ext_merged_new['lagged_Offer_Date'] = (ext_merged_new['Offer_Date'] - pd.Timedelta(days=1)).dt.normalize()


In [44]:
# ===========================================
# 9. Align Offer Dates to Nearest Previous NYSE Trading Day
# ===========================================


# Step 1: Get official NYSE trading calendar (business days only)
nyse = mcal.get_calendar('NYSE')
trading_days = nyse.schedule(start_date='2009-01-01', end_date='2023-12-31')

# Step 2: Convert to flat daily list of trading dates
trading_days = mcal.date_range(trading_days, frequency='1D')
trading_days_df = pd.DataFrame({'Trading_Day': trading_days})

# Step 3: Normalize timezone and ensure sorting
trading_days_df['Trading_Day'] = pd.to_datetime(trading_days_df['Trading_Day']).dt.tz_localize(None)
trading_days_df['Trading_Day'] = trading_days_df['Trading_Day'].dt.normalize()
trading_days_df = trading_days_df.sort_values('Trading_Day')

# Step 4: Use merge_asof to align each Offer_Date to the previous valid NYSE trading day
merged_df = pd.merge_asof(
    ext_merged_new.sort_values('Offer_Date'),
    trading_days_df,
    left_on='Offer_Date',
    right_on='Trading_Day',
    direction='backward'  # gets the most recent trading day <= offer date
)

# Step 5: Update lagged_Offer_Date using adjusted trading day
merged_df['lagged_Offer_Date'] = merged_df['Trading_Day']

# Step 6: Clean up helper column and restore original sort
ext_merged_new = merged_df.drop(columns=['Trading_Day']).sort_values('Issue_Date')

# Final check on DataFrame structure
ext_merged_new.info()


<class 'pandas.core.frame.DataFrame'>
Index: 13755 entries, 0 to 13754
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag      111

## Working on Annual Ratios
This section enriches the SEO dataset with company-level financial ratios (e.g., ROA, EPS) by merging with external annual report data using ticker and CUSIP9 mappings. It includes:

- Cleaning and harmonizing CUSIP formats

- Handling duplicate financial records

- Merging on fiscal quarter and year for temporal alignment

In [45]:
# ===========================================
# 1. Load Annual Ratios Based on Ticker
# ===========================================

# Load ratios generated using issue ticker ('tic')
file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\annual_ratios_issue_ticker.zip'
with zipfile.ZipFile(file_path) as z:
    with z.open('ow6jhpsol4tsz6tv.csv') as f:
        ratio_tic = pd.read_csv(f)

ratio_tic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301993 entries, 0 to 301992
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   permno           301993 non-null  int64  
 1   adate            300330 non-null  object 
 2   qdate            301993 non-null  object 
 3   public_date      301993 non-null  object 
 4   pe_exi           295735 non-null  float64
 5   pe_inc           295739 non-null  float64
 6   npm              279404 non-null  float64
 7   opmbd            279391 non-null  float64
 8   gpm              277685 non-null  float64
 9   cfm              278678 non-null  float64
 10  roa              299001 non-null  float64
 11  roe              279231 non-null  float64
 12  cash_conversion  195705 non-null  float64
 13  TICKER           300086 non-null  object 
 14  cusip            300319 non-null  object 
dtypes: float64(9), int64(1), object(5)
memory usage: 34.6+ MB


In [46]:
# ===========================================
# 2. Load Annual Ratios Based on CUSIP9
# ===========================================

file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\annual_ratios_complete_cusip9.zip'
with zipfile.ZipFile(file_path) as z:
    with z.open('l7ao2giuola6iaee.csv') as f:
        ratio_cusip9 = pd.read_csv(f)

ratio_cusip9.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293974 entries, 0 to 293973
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   permno           293974 non-null  int64  
 1   adate            292302 non-null  object 
 2   qdate            293974 non-null  object 
 3   public_date      293974 non-null  object 
 4   pe_exi           288073 non-null  float64
 5   pe_inc           288073 non-null  float64
 6   npm              271323 non-null  float64
 7   opmbd            271312 non-null  float64
 8   gpm              269528 non-null  float64
 9   cfm              270551 non-null  float64
 10  roa              291115 non-null  float64
 11  roe              270767 non-null  float64
 12  cash_conversion  187243 non-null  float64
 13  TICKER           292038 non-null  object 
 14  cusip            292267 non-null  object 
dtypes: float64(9), int64(1), object(5)
memory usage: 33.6+ MB


In [47]:
# ===========================================
# 3. Merge CUSIP9-Based Ratios Not Already in Ticker-Based Ratios
# ===========================================

# Filter out rows from CUSIP9-based ratios that aren't already in ticker-based ratios
unique_cusip_rows = ratio_cusip9[~ratio_cusip9['cusip'].isin(ratio_tic['cusip'])]

# Append them to the main ratio_tic dataframe
ratio_tic = pd.concat([ratio_tic, unique_cusip_rows], ignore_index=True)
ratio_tic = ratio_tic.reset_index(drop=True)
ratio_tic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316457 entries, 0 to 316456
Data columns (total 15 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   permno           316457 non-null  int64  
 1   adate            314658 non-null  object 
 2   qdate            316457 non-null  object 
 3   public_date      316457 non-null  object 
 4   pe_exi           309751 non-null  float64
 5   pe_inc           309755 non-null  float64
 6   npm              292977 non-null  float64
 7   opmbd            292964 non-null  float64
 8   gpm              291163 non-null  float64
 9   cfm              292144 non-null  float64
 10  roa              313269 non-null  float64
 11  roe              292155 non-null  float64
 12  cash_conversion  204257 non-null  float64
 13  TICKER           314544 non-null  object 
 14  cusip            314783 non-null  object 
dtypes: float64(9), int64(1), object(5)
memory usage: 36.2+ MB


In [48]:
# ===========================================
# 4. Convert 8-digit CUSIP to 9-digit CUSIP
# ===========================================

# Helper function to calculate check digit and generate CUSIP9
def calculate_cusip_check_digit(cusip_8):
    if not isinstance(cusip_8, str) or len(cusip_8) != 8:
        return None
    def char_to_value(c):
        if c.isdigit(): return int(c)
        elif c.isalpha(): return ord(c.upper()) - ord('A') + 10
        elif c == '*': return 36
        elif c == '@': return 37
        elif c == '#': return 38
        return None
    total = 0
    for i in range(8):
        val = char_to_value(cusip_8[i])
        if val is None: return None
        weight = 2 if (i + 1) % 2 == 0 else 1
        val *= weight
        total += (val // 10) + (val % 10)
    check_digit = (10 - (total % 10)) % 10
    return cusip_8 + str(check_digit)

# Apply conversion to get cusip_9
ratio_tic['cusip_9'] = ratio_tic['cusip'].apply(calculate_cusip_check_digit)

# Check for failed conversions
failed_conversions = ratio_tic[ratio_tic['cusip_9'].isnull()]
failed_conversions.reset_index(drop=True, inplace=True)
failed_conversions.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1674 entries, 0 to 1673
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   permno           1674 non-null   int64  
 1   adate            1600 non-null   object 
 2   qdate            1674 non-null   object 
 3   public_date      1674 non-null   object 
 4   pe_exi           1526 non-null   float64
 5   pe_inc           1526 non-null   float64
 6   npm              1255 non-null   float64
 7   opmbd            1251 non-null   float64
 8   gpm              1233 non-null   float64
 9   cfm              1252 non-null   float64
 10  roa              1555 non-null   float64
 11  roe              778 non-null    float64
 12  cash_conversion  814 non-null    float64
 13  TICKER           0 non-null      object 
 14  cusip            0 non-null      object 
 15  cusip_9          0 non-null      object 
dtypes: float64(9), int64(1), object(6)
memory usage: 209.4+ KB


In [49]:
# ===========================================
# 5. Prepare for Merging by Creating Lagged Dates
# ===========================================

# Convert public_date to datetime
ratio_tic['public_date'] = pd.to_datetime(ratio_tic['public_date'], errors='coerce')

# Add 1 month to shift to reporting quarter
ratio_tic['lagged_date'] = ratio_tic['public_date'] + pd.DateOffset(months=1)

# Drop irrelevant columns
ratio_tic.drop(columns=['adate', 'qdate', 'cusip'], inplace=True)
ratio_tic.reset_index(drop=True, inplace=True)
ratio_tic.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316457 entries, 0 to 316456
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   permno           316457 non-null  int64         
 1   public_date      316457 non-null  datetime64[ns]
 2   pe_exi           309751 non-null  float64       
 3   pe_inc           309755 non-null  float64       
 4   npm              292977 non-null  float64       
 5   opmbd            292964 non-null  float64       
 6   gpm              291163 non-null  float64       
 7   cfm              292144 non-null  float64       
 8   roa              313269 non-null  float64       
 9   roe              292155 non-null  float64       
 10  cash_conversion  204257 non-null  float64       
 11  TICKER           314544 non-null  object        
 12  cusip_9          314783 non-null  object        
 13  lagged_date      316457 non-null  datetime64[ns]
dtypes: datetime64[ns](2)

In [50]:
# ===========================================
# 6. Extract Quarter and Year for Joining Keys
# ===========================================

# From lagged_date (used in financial reports)
ratio_tic['quarter_value'] = (ratio_tic['lagged_date'].dt.month - 1) // 3 + 1
ratio_tic['year_value'] = ratio_tic['lagged_date'].dt.year

# From Issue_Date (SEO timing)
ext_merged_new['quarter_seo'] = (ext_merged_new['Issue_Date'].dt.month - 1) // 3 + 1
ext_merged_new['year_seo'] = ext_merged_new['Issue_Date'].dt.year

ext_merged_new.info()


<class 'pandas.core.frame.DataFrame'>
Index: 13755 entries, 0 to 13754
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag      111

In [51]:
# ===========================================
# 7. Merge Annual Ratios with SEO Dataset
# ===========================================

# Initial merge using quarter/year and CUSIP9
sar = pd.merge(
    ext_merged_new,
    ratio_tic,
    how='left',
    left_on=['quarter_seo', 'year_seo', 'Issuer_CUSIP9'],
    right_on=['quarter_value', 'year_value', 'cusip_9']
)

# Drop temporary merge keys
sar.drop(columns=['quarter_seo', 'year_seo', 'quarter_value', 'year_value'], inplace=True)
sar.reset_index(drop=True, inplace=True)
sar.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27127 entries, 0 to 27126
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 27127 non-null  object        
 1   Price_Date_Diff               27127 non-null  int64         
 2   Issue_Date                    27127 non-null  datetime64[ns]
 3   Offer_Year                    27127 non-null  int64         
 4   New_Issues_Primary_Exchange   27018 non-null  category      
 5   Issuer_Name_Full              27127 non-null  object        
 6   Issuer_CUSIP8                 24986 non-null  object        
 7   Issuer_CUSIP9                 27127 non-null  object        
 8   New_Issues_HiTech_Group       27127 non-null  object        
 9   Issue_Type                    27127 non-null  category      
 10  Regulation_Types              25350 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

In [52]:
# ===========================================
# 8. Handle Duplicate Entries in Annual Ratios
# ===========================================

# Check for rows that caused inflation due to duplicates
duplicates = ratio_tic.duplicated(subset=['quarter_value', 'year_value', 'cusip_9'], keep=False)
print(ratio_tic[duplicates])

# Drop duplicates, keeping the most recent (last)
ratio_tic = ratio_tic.drop_duplicates(subset=['quarter_value', 'year_value', 'cusip_9'], keep='last')


        permno public_date  pe_exi  pe_inc    npm  opmbd    gpm    cfm    roa  \
0        10001  2009-12-31  13.553  13.553  0.042  0.127  0.127  0.069  0.146   
1        10001  2010-01-31  13.237  13.237  0.042  0.127  0.127  0.069  0.146   
2        10001  2010-02-28   6.334   6.334  0.095  0.158  0.158  0.132  0.146   
3        10001  2010-03-31   6.437   6.437  0.095  0.158  0.158  0.132  0.146   
4        10001  2010-04-30   7.209   7.209  0.095  0.158  0.158  0.132  0.146   
...        ...         ...     ...     ...    ...    ...    ...    ...    ...   
316452   93435  2012-01-31   0.651   0.651  0.385  0.294  0.393  0.411  0.314   
316453   93435  2012-02-29   1.095   1.095  0.314  0.258  0.345  0.344  0.231   
316454   93435  2012-03-31   1.964   1.964  0.314  0.258  0.345  0.344  0.231   
316455   93435  2012-04-30   1.285   1.285  0.314  0.258  0.345  0.344  0.231   
316456   93435  2012-05-31   1.052   1.052  0.237  0.234  0.325  0.269  0.190   

          roe  cash_convers

In [53]:
# ===========================================
# 9. Re-Merge with Deduplicated Ratios and Final Clean-up
# ===========================================

# Re-run merge
sar = pd.merge(
    ext_merged_new,
    ratio_tic,
    how='left',
    left_on=['quarter_seo', 'year_seo', 'Issuer_CUSIP9'],
    right_on=['quarter_value', 'year_value', 'cusip_9']
)

# Drop temporary columns and rename lagged date for clarity
sar.drop(columns=['quarter_seo', 'year_seo', 'quarter_value', 'year_value'], inplace=True)
sar.reset_index(drop=True, inplace=True)
sar.rename(columns={'lagged_date': 'Public_Company_Lag_Date'}, inplace=True)

sar.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker                 13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

## Working on merging EPS, Dividend Data based on lagged offer Date

In [54]:
# ===========================================
# 1. Load Daily EPS and Dividend Data by Ticker
# ===========================================

# Load dataset with daily fundamentals based on ticker
file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\security_data_using_issue_ticker.zip'
with zipfile.ZipFile(file_path) as z:
    with z.open('meabriv7ju9f5juy.csv') as f:
        eps_tic = pd.read_csv(f)

eps_tic.info()


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\268941558.py:9: DtypeWarning: Columns (5,15) have mixed types. Specify dtype option on import or set low_memory=False.
  eps_tic = pd.read_csv(f)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7773604 entries, 0 to 7773603
Data columns (total 17 columns):
 #   Column    Dtype  
---  ------    -----  
 0   GVKEY     int64  
 1   LPERMNO   int64  
 2   iid       int64  
 3   datadate  object 
 4   tic       object 
 5   cusip     object 
 6   cshoc     float64
 7   cshtrd    float64
 8   dvi       float64
 9   eps       float64
 10  prccd     float64
 11  trfd      float64
 12  cik       float64
 13  idbflag   object 
 14  sic       int64  
 15  spcsrc    object 
 16  state     object 
dtypes: float64(7), int64(4), object(6)
memory usage: 1008.2+ MB


In [55]:
# ===========================================
# 2. Load Daily EPS and Dividend Data by CUSIP9
# ===========================================
file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\security_daily_complete_cusip9.zip'
with zipfile.ZipFile(file_path) as z:
    with z.open('qgcrmhfqvztoeuk6.csv') as f:
        eps_cusip = pd.read_csv(f)

eps_cusip.info()


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\1187154460.py:7: DtypeWarning: Columns (5,15) have mixed types. Specify dtype option on import or set low_memory=False.
  eps_cusip = pd.read_csv(f)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6775539 entries, 0 to 6775538
Data columns (total 17 columns):
 #   Column    Dtype  
---  ------    -----  
 0   GVKEY     int64  
 1   LPERMNO   int64  
 2   iid       int64  
 3   datadate  object 
 4   tic       object 
 5   cusip     object 
 6   cshoc     float64
 7   cshtrd    float64
 8   dvi       float64
 9   eps       float64
 10  prccd     float64
 11  trfd      float64
 12  cik       float64
 13  idbflag   object 
 14  sic       int64  
 15  spcsrc    object 
 16  state     object 
dtypes: float64(7), int64(4), object(6)
memory usage: 878.8+ MB


In [56]:
# ===========================================
# 3. Combine Both Sources into a Unified Daily Dataset
# ===========================================
# Combine ticker and CUSIP-based data
tic_cusip_daily = pd.concat([eps_tic, eps_cusip], ignore_index=True)

# Convert 'datadate' to datetime for temporal merging
tic_cusip_daily['datadate'] = pd.to_datetime(tic_cusip_daily['datadate'], errors='coerce')

tic_cusip_daily.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14549143 entries, 0 to 14549142
Data columns (total 17 columns):
 #   Column    Dtype         
---  ------    -----         
 0   GVKEY     int64         
 1   LPERMNO   int64         
 2   iid       int64         
 3   datadate  datetime64[ns]
 4   tic       object        
 5   cusip     object        
 6   cshoc     float64       
 7   cshtrd    float64       
 8   dvi       float64       
 9   eps       float64       
 10  prccd     float64       
 11  trfd      float64       
 12  cik       float64       
 13  idbflag   object        
 14  sic       int64         
 15  spcsrc    object        
 16  state     object        
dtypes: datetime64[ns](1), float64(7), int64(4), object(5)
memory usage: 1.8+ GB


In [57]:
# ===========================================
# 4. Merge Daily EPS/DVI with SEO Dataset Based on Lagged Offer Date
# ===========================================

# Create unique ID to preserve row identity during multiple merges
sar['iD'] = range(1, len(sar) + 1)

# Prepare column names to align with sar
tic_cusip_daily = tic_cusip_daily.rename(columns={
    'datadate': 'lagged_Offer_Date',
    'cusip': 'Issuer_CUSIP9',
    'tic': 'Issuer_Ticker'
})

# First merge on lagged_offer_date + CUSIP9
merged1 = pd.merge(sar, tic_cusip_daily, how='left', 
                   on=['lagged_Offer_Date', 'Issuer_CUSIP9']).drop_duplicates()

# Second merge on lagged_offer_date + Ticker
merged2 = pd.merge(sar, tic_cusip_daily, how='left', 
                   on=['lagged_Offer_Date', 'Issuer_Ticker']).drop_duplicates()

# Filter to keep only additional columns from the second merge
merged2 = merged2.drop_duplicates(subset=['iD'], keep='last')
merged2 = merged2[['iD', 'cshoc', 'cshtrd', 'dvi', 'eps', 'prccd', 'trfd', 'cik', 'idbflag', 'sic', 'spcsrc', 'state']]

# Merge both results back on iD
final_merged = pd.merge(merged1, merged2, on='iD', how='left')
final_merged.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13755 entries, 0 to 13754
Data columns (total 67 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Issuer_Ticker_x               13755 non-null  object        
 1   Price_Date_Diff               13755 non-null  int64         
 2   Issue_Date                    13755 non-null  datetime64[ns]
 3   Offer_Year                    13755 non-null  int64         
 4   New_Issues_Primary_Exchange   13678 non-null  category      
 5   Issuer_Name_Full              13755 non-null  object        
 6   Issuer_CUSIP8                 11902 non-null  object        
 7   Issuer_CUSIP9                 13755 non-null  object        
 8   New_Issues_HiTech_Group       13755 non-null  object        
 9   Issue_Type                    13755 non-null  category      
 10  Regulation_Types              12429 non-null  object        
 11  VC_Backed_IPO_Issue_Flag    

In [58]:
# ===========================================
# 5. Fill Missing Values and Clean Up Columns
# ===========================================
columns = ['cshoc', 'cshtrd', 'dvi', 'eps', 'prccd', 'trfd', 'cik', 'idbflag', 'sic', 'spcsrc', 'state']

# Fill missing _x columns with _y values from second merge
for col in columns:
    final_merged[f'{col}_x'] = final_merged[f'{col}_x'].fillna(final_merged[f'{col}_y'])

# Drop _y columns
final_merged.drop(columns=[f'{col}_y' for col in columns], inplace=True)

# Drop unnecessary metadata columns
final_merged.drop(columns=['public_date', 'cusip_9', 'LPERMNO', 'iid', 'prccd_x', 'GVKEY', 
                           'iD', 'permno', 'TICKER', 'Issuer_Ticker_y'], inplace=True)


In [59]:
# ===========================================
# 6. Rename Final Output Columns for Clarity
# ===========================================
complete_seo = final_merged.rename(columns={
    'eps_x': 'Current_EPS',
    'dvi_x': 'Annual_Dividend',
    'cshtrd_x': 'Trading_Volume',
    'cshoc_x': 'Shares_Outstanding',
    'spcsrc_x': 'S_P_Quality_Ranking',
    'sic_x': 'Industry_Classification_Code',
    'idbflag_x': 'International_Domestic',
    'trfd_x': 'Total_Return_Factor',
    'pe_exi': 'P_E_Excl_Extra',
    'pe_inc': 'P_E_Incl_Extra',
    'npm': 'Net_Profit_Margin',
    'opmbd': 'Operating_Profit_Margin_Before_Depreciation',
    'gpm': 'Gross_Profit_Margin',
    'cfm': 'Cash_Flow_Margin',
    'roa': 'Return_on_Assets',
    'roe': 'Return_on_Equity',
    'cash_conversion': 'Cash_Conversion_Cycle_Days',
    'cik_x': 'cik',
    'state_x': 'state',
    'Issuer_Ticker_x': 'Ticker'
})


## Reading US Treasury and Inflation from WRDS

Variable Name	Type	Description<br>
B30	Char	30 Year Bond (B30)<br>
B20	Char	20 Year Bond (B20)<br>
B10	Char	10 Year Bond (B10)<br>
B7	Char	7 Year Bond (B7)<br>
B5	Char	5 Year Bond (B5)<br>
B2	Char	2 Year Bond (B2)<br>
B1	Char	1 Year Bond (B1)<br>
T90	Char	90 Day Bill (T90)<br>
T30	Char	30 Day Bill (T30)<br>
CPI	Float	Inflation (CPI)<br>

In [60]:
# ===========================================
# 7. Load US Treasury Yield and CPI Dataset
# ===========================================
file_path = r'data\Updated SEO ISSUES Dataset\Wharton Datasets\US_treasury_and_inflation_index.zip'
with zipfile.ZipFile(file_path) as z:
    with z.open('grm77wxtnzk6lofb.csv') as f:
        uti = pd.read_csv(f)

# Convert date format and extract month/year
uti['caldt'] = pd.to_datetime(uti['caldt'])
uti['month'] = uti['caldt'].dt.month
uti['year'] = uti['caldt'].dt.year

# Extract same from SEO data
complete_seo['month'] = complete_seo['lagged_Offer_Date'].dt.month
complete_seo['year'] = complete_seo['lagged_Offer_Date'].dt.year

# Merge on month-year
merged_uti = pd.merge(
    complete_seo,
    uti[['month', 'year', 'b1ret', 't30ret', 'cpiret']],
    on=['month', 'year'],
    how='left'
)

# Drop temp columns and rename
merged_uti.drop(columns=['month', 'year'], inplace=True)
merged_uti.rename(columns={
    'b1ret': 'tbill_30_return',
    't30ret': 'tbond_30_return',
    'cpiret': 'consumer_price_index'
}, inplace=True)


## US treasury and inflation dataset column definitions
| Column    | Full Name                        | Description                                                                 |
|-----------|----------------------------------|-----------------------------------------------------------------------------|
| `caldt`   | Quotation Date                   | The date on which the price or yield data was recorded.                     |
| `b1ret`   | 1-Year Treasury Bill Return      | Return on a 1-Year U.S. Treasury bill, expressed as a percentage.           |
| `b1ind`   | 1-Year Treasury Bill Index       | Index level for 1-Year Treasury bill, base investment of $100 on 12/29/1972.|
| `t90ret`  | 90-Day Treasury Bill Return      | Return on a 90-day U.S. Treasury bill, expressed as a percentage.           |
| `t90ind`  | 90-Day Treasury Bill Index       | Index level for 90-day Treasury bill, base investment of $100 on 12/29/1972.|
| `t30ret`  | 30-Day Treasury Bond Return     | Return on a 30-Day U.S. Treasury bond, expressed as a percentage.          |
| `t30ind`  | 30-Day Treasury Bond Index      | Index level for 30-Day Treasury bond, base investment of $100 on 12/29/1972.|
| `cpiret`  | Consumer Price Index Return      | Return on the Consumer Price Index (CPI), measuring average price changes.  |
| `cpiind`  | Consumer Price Index Level       | Index level of the CPI, usually based to 100 in a base year.                |


In [61]:
# ===========================================
# 8. Load and Align Projected GDP Growth Rate
# ===========================================
pgr = pd.read_csv(r'data\Updated SEO ISSUES Dataset\Wharton Datasets\Projected growth rate.csv')

# Convert and reindex to daily granularity
pgr['observation_date'] = pd.to_datetime(pgr['observation_date'])
pgr = pgr.set_index('observation_date').sort_index()
pgr = pgr.reindex(pd.date_range(pgr.index.min(), pgr.index.max(), freq='D'))
pgr.fillna(method='ffill', inplace=True)
pgr.reset_index(inplace=True)

# Rename columns for merge
pgr.rename(columns={'GDPC1CTMLR': 'proj_growth_rate', 'index': 'date'}, inplace=True)

# Merge with main dataset
merged_pgr = merged_uti.merge(
    pgr,
    how='left',
    left_on='lagged_Offer_Date',
    right_on='date'
)

# Clean up
merged_pgr.drop(columns=['date'], inplace=True)


C:\Users\sjen591\AppData\Local\Temp\ipykernel_7012\1490202134.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  pgr.fillna(method='ffill', inplace=True)


In [62]:
# ===========================================
# 9. Export Final Dataset to CSV and Parquet
# ===========================================
import os
import pyarrow

output_path = r'data\Updated SEO ISSUES Dataset'
csv_filename = r'SEO_ISSUES_After_ratios_and_pe.csv'
parquet_filename = r'SEO_ISSUES_After_ratios_and_pe.parquet'

# Export both formats
merged_pgr.to_csv(os.path.join(output_path, csv_filename), index=False)
merged_pgr.to_parquet(os.path.join(output_path, parquet_filename), index=False)
